In [1]:
!pip install mne
!pip install neural-structured-learning

     |████████████████████████████████| 6.6MB 4.8MB/s 
     |████████████████████████████████| 122kB 7.0MB/s 


In [2]:
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
# from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import sys
import tensorflow as tf
from tensorflow.keras import layers, Model, models
from tensorflow.keras.metrics import BinaryAccuracy
from tensorflow.keras.optimizers import Adam
from tensorflow_addons.metrics import CohenKappa
from tensorflow.keras.constraints import max_norm
import neural_structured_learning as nsl
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [3]:
def band_pass_filter(eeg, freq_range):
  info = mne.create_info(22, 250, ch_types=["eeg"] * 22)
  raw = mne.io.RawArray(eeg.T, info)
  raw.filter(freq_range[0], freq_range[1], fir_design='firwin')

  return raw._data.T

# def band_pass_filter(eeg, freq_range):
#     info = mne.create_info(22, 250, ch_types=["eeg"] * 22)
#     raw = mne.io.RawArray(eeg.T, info)
    
#     iir_params = dict(order=5, ftype='cheby2', rs=2.)
#     raw = raw.filter(freq_range[0], freq_range[1], fir_design='firwin', method='iir', iir_params=iir_params)

#     return raw._data.T
  
def load_data(mode='train', fno = 1):
  if fno != 4:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

  else:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

In [4]:
def drop_classes(df):
  i = 0
  indexes_to_drop = []
  while i < len(df):
    if(df['class'][i]==4):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    elif(df['class'][i]==3):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    else:
      i += 1

  indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
  df_sliced = df.take(list(indexes_to_keep))

  df_sliced = df_sliced.reset_index(drop=True)
  return df_sliced

In [5]:
def convert_class(sfreq, trigger_points, df):
  asynch_label = []
  i = 0
  while i < len(df['class']):
      if(df['class'][i] == 1):
          end = (sfreq * trigger_points[1]) + 1
          for j in range(i, i+end):
              asynch_label.append(1)
          i += end
      elif(df['class'][i] == 2):
          end = (sfreq * trigger_points[2]) + 1
          for j in range(i, i+end):
              asynch_label.append(2)
          i += end
      elif(df['class'][i] == 3):
          end = (sfreq * trigger_points[3]) + 1
          for j in range(i, i+end):
              asynch_label.append(3)
          i += end
      elif(df['class'][i] == 4):
          end = (sfreq * trigger_points[4]) + 1
          for j in range(i, i+end):
              asynch_label.append(4)
          i += end
      elif(df['class'][i] == 0):
          asynch_label.append(0)
          i += 1

  return np.array(asynch_label)

def prune_records(data_new, label):
  train_data = []
  label_data = []

  for i in range(len(label)):
      if label[i] != 0:
          label_data.append(label[i])
          train_data.append(data_new[i])
  label_data = np.array(label_data)
  train_data = np.array(train_data)

  return train_data, label_data

In [6]:
def data_win(sfreq, data, asynch_label):
  sampling_window = 2 * sfreq
  shift_length = 1 * sfreq
  t_start = 0
    
  new_data = []
  labels = []


  while t_start + sampling_window < data.shape[0]:
      new_data.append(data[t_start:t_start+sampling_window, :].T)
      labels.append(asynch_label[t_start:t_start+sampling_window])
    
      t_start = t_start + shift_length
    
  return np.array(new_data), np.array(labels)

def transform_label(label_new):
  label = []

  for i in label_new:
      count1 = np.count_nonzero(i==1)
      count2 = np.count_nonzero(i==2)
      count3 = np.count_nonzero(i==3)
      count4 = np.count_nonzero(i==4)
      if count1 >= 250:
          to_add = 1
      elif count2 >= 250:
          to_add = 2
      elif count3 >= 250:
          to_add = 3
      elif count4 >= 250:
          to_add = 4
      else:
          to_add = 0
      label.append(to_add)
      
  label = np.array(label)

  return label

In [7]:
def get_train_data(fno=1):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}
  freq_range = [8, 32]

  df = load_data(mode='train', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)

  out_data = band_pass_filter(train_df.iloc[:, :-1].values, freq_range=freq_range)

  X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=asynch_label)
  y = transform_label(y)
  X, y = prune_records(X, y)
  dim1, dim2, dim3 = X.shape
  X_new = X.reshape((dim1, 1, dim2, dim3))
  y = y-1

  return X_new, y


def get_eval_data(fno=1):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}
  freq_range = [8, 32]

  df = load_data(mode='test', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)

  out_data = band_pass_filter(train_df.iloc[:, :-1].values, freq_range=freq_range)

  X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=asynch_label)
  y = transform_label(y)
  X, y = prune_records(X, y)
  dim1, dim2, dim3 = X.shape
  X_eval = X.reshape((dim1, 1, dim2, dim3))
  y = y - 1

  return X_eval, y

In [9]:
def EEGNET(channels=22, samples=500):
  input1 = layers.Input(shape=(1, channels, samples), name='input')
  b1 = layers.Conv2D(8, (1, 250), padding='same', use_bias=False, data_format='channels_first')(input1)
  b1 = layers.BatchNormalization(axis=1)(b1)
  b1 = layers.DepthwiseConv2D((channels, 1), use_bias=False, depth_multiplier=2, depthwise_constraint=max_norm(1.), data_format='channels_first')(b1)
  b1 = layers.BatchNormalization(axis=1)(b1)
  b1 = layers.Activation('elu')(b1)
  b1 = layers.AveragePooling2D((1, 4), data_format='channels_first')(b1)
  b1 = layers.Dropout(0.5)(b1)

  b2 = layers.SeparableConv2D(16, (1, 16), padding='same', use_bias=False, data_format='channels_first')(b1)
  b2 = layers.BatchNormalization(axis=1)(b2)
  b2 = layers.Activation('elu')(b2)
  b2 = layers.AveragePooling2D((1, 8), data_format='channels_first')(b2)
  b2 = layers.Dropout(0.5)(b2)

  flatten = layers.Flatten()(b2)

  dense = layers.Dense(1, kernel_constraint=max_norm(0.25))(flatten)
  activation = layers.Activation('sigmoid')(dense)

  eegnet = Model(inputs = input1, outputs=activation)

  return eegnet

In [10]:
from numpy.random import seed
seed(1)
# from tensorflow import set_random_seed
# set_random_seed(2)

In [11]:
ba = BinaryAccuracy()
kappa = CohenKappa(num_classes=2)

In [13]:
def network_train(X_train, y_train, X_eval, y_eval):
  X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

  sum_train_acc = []
  sum_test_acc = []
  sum_train_kappa = []
  sum_test_kappa = []

  times = 50

  for i in range(times):
    eegnet = EEGNET(channels=22, samples=500)

    adv_config = nsl.configs.make_adv_reg_config(multiplier=0.2, adv_step_size=0.001)
    adv_model = nsl.keras.AdversarialRegularization(eegnet, adv_config=adv_config)

    my_callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    ]

    adv_model.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=[ba, kappa])

    history = adv_model.fit({'input': X_train, 'label': y_train}, batch_size=40, epochs=500, shuffle=True,
                            validation_data={'input': X_test, 'label': y_test},
                            verbose=2, callbacks=my_callbacks)
    # adv_model.evaluate({'feature': x_test, 'label': y_test})

    # eegnet.compile(optimizer=adam, loss='binary_crossentropy', metrics=[ba, kappa])

    # history = eegnet.fit(X_train, y_train, batch_size=25, epochs=500, validation_split=0.25, verbose=2, callbacks=my_callbacks)

    sum_train_acc.append(np.max(history.history['val_binary_accuracy']))
    sum_train_kappa.append(np.max(history.history['val_cohen_kappa']))

    # score = eegnet.evaluate(X_eval, y_eval, batch_size=5)
    score = adv_model.evaluate({'input': X_eval, 'label': y_eval}, batch_size=40)
    sum_test_acc.append(score[2])
    sum_test_kappa.append(score[3])
    print(score)


    train_acc = []
    train_kappa = []
    test_acc = []
    test_kappa = []

    train_acc.append([np.mean(sum_train_acc), np.std(sum_train_acc)])
    train_kappa.append([np.mean(sum_train_kappa), np.std(sum_train_kappa)])
    test_acc.append([np.mean(sum_test_acc), np.std(sum_test_acc)])
    test_kappa.append([np.mean(sum_test_kappa), np.std(sum_test_kappa)])

  print('Train Accuracy --->', train_acc)
  print('Train Kappa --->', train_kappa)
  print('Test Accuracy --->', test_acc)
  print('Test Kappa --->', test_kappa)


In [ ]:
X_train, y_train = get_train_data(fno=1)
X_eval, y_eval = get_eval_data(fno=1)
network_train(X_train, y_train, X_eval, y_eval)

In [15]:
  # sum_train_acc = []
# sum_test_acc = []
# sum_train_kappa = []
# sum_test_kappa = []

# times = 50

# for i in range(times):
#   eegnet = EEGNET(channels=22, samples=500)

#   adv_config = nsl.configs.make_adv_reg_config(multiplier=0.2, adv_step_size=0.001)
#   adv_model = nsl.keras.AdversarialRegularization(eegnet, adv_config=adv_config)

#   my_callbacks = [
#       EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
#   ]

#   adv_model.compile(optimizer='adam',
#                   loss='binary_crossentropy',
#                   metrics=[ba, kappa])

#   history = adv_model.fit({'input': X_train, 'label': y_train}, batch_size=40, epochs=500, shuffle=True,
#                           validation_data={'input': X_test, 'label': y_test},
#                           verbose=2, callbacks=my_callbacks)
#   # adv_model.evaluate({'feature': x_test, 'label': y_test})

#   # eegnet.compile(optimizer=adam, loss='binary_crossentropy', metrics=[ba, kappa])

#   # history = eegnet.fit(X_train, y_train, batch_size=25, epochs=500, validation_split=0.25, verbose=2, callbacks=my_callbacks)

#   sum_train_acc.append(np.max(history.history['val_binary_accuracy']))
#   sum_train_kappa.append(np.max(history.history['val_cohen_kappa']))

#   # score = eegnet.evaluate(X_eval, y_eval, batch_size=5)
#   score = adv_model.evaluate({'input': X_eval, 'label': y_eval})
#   sum_test_acc.append(score[2])
#   sum_test_kappa.append(score[3])
#   print(score)


#   train_acc = []
#   train_kappa = []
#   test_acc = []
#   test_kappa = []

#   train_acc.append([np.mean(sum_train_acc), np.std(sum_train_acc)])
#   train_kappa.append([np.mean(sum_train_kappa), np.std(sum_train_kappa)])
#   test_acc.append([np.mean(sum_test_acc), np.std(sum_test_acc)])
#   test_kappa.append([np.mean(sum_test_kappa), np.std(sum_test_kappa)])

# print('Train Accuracy --->', train_acc)
# print('Train Kappa --->', train_kappa)
# print('Test Accuracy --->', test_acc)
# print('Test Kappa --->', test_kappa)

In [ ]:
X_train, y_train = get_train_data(fno=2)
X_eval, y_eval = get_eval_data(fno=2)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=3)
X_eval, y_eval = get_eval_data(fno=3)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=4)
X_eval, y_eval = get_eval_data(fno=4)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=5)
X_eval, y_eval = get_eval_data(fno=5)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=6)
X_eval, y_eval = get_eval_data(fno=6)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=7)
X_eval, y_eval = get_eval_data(fno=7)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=8)
X_eval, y_eval = get_eval_data(fno=8)
network_train(X_train, y_train, X_eval, y_eval)

In [ ]:
X_train, y_train = get_train_data(fno=9)
X_eval, y_eval = get_eval_data(fno=9)
network_train(X_train, y_train, X_eval, y_eval)